# Day 045 — Exercise 1: extract

**What you'll build:** `extract(csv_text: str) -> list[dict]` — parse a CSV string into a list of dicts using `csv.DictReader` and `io.StringIO`.

**Why it matters:** The first step of every ETL pipeline is getting raw data into a Python-native format. `csv.DictReader` reads each row as a dict keyed by the CSV header. `io.StringIO` wraps a string as a file-like object so `DictReader` can read from it without touching the filesystem. All values remain strings after extraction — type conversion happens in the Transform step.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import csv
import io
from sqlalchemy import create_engine, String, Float, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = 'sales'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    date:     Mapped[str]   = mapped_column(String(20))
    product:  Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    amount:   Mapped[float] = mapped_column()
    region:   Mapped[str]   = mapped_column(String(50))

    def __repr__(self):
        return f'Sale(id={self.id}, product={self.product!r}, amount={self.amount})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


CSV_SOURCE = (
    'date,product,category,amount,region\n'
    '2024-01-15,Laptop,Electronics,999.99,East\n'
    '2024-01-16,Headphones,Electronics,149.99,West\n'
    '2024-01-17,Desk Chair,Furniture,349.00,East\n'
    '2024-01-18,,Furniture,199.00,North\n'
    '2024-01-19,Pen Set,Stationery,twelve,South\n'
    '2024-01-20,Monitor,Electronics,599.99,West\n'
    '2024-01-21,Keyboard,Electronics,79.99,East\n'
    '2024-01-22,Webcam,Electronics,,North\n'
    '2024-01-23,Lamp,Furniture,45.99,South\n'
    '2024-01-24,Notebook,Stationery,8.99,West\n'
)

## Your Implementation

In [ ]:
def extract(csv_text: str) -> list:
    """
    Parse CSV text into a list of dicts.

    Use csv.DictReader(io.StringIO(csv_text)) to parse the text.
    Return list(reader) — each row becomes a dict keyed by column name.
    All values are strings at this stage; type conversion happens later.
    """
    # TODO: reader = csv.DictReader(io.StringIO(csv_text))
    # TODO: return list(reader)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'extract' in globals()
        passed += 1; print('\u2705 Check 1: extract is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a list
    try:
        result = extract(CSV_SOURCE)
        assert isinstance(result, list), \
            f'expected list, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a list')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: correct row count (header excluded)
    try:
        assert len(result) == 10, \
            f'expected 10 rows, got {len(result)}'
        passed += 1; print(f'\u2705 Check 3: 10 rows extracted')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: rows are dicts with correct keys
    try:
        expected_keys = {'date', 'product', 'category', 'amount', 'region'}
        assert all(isinstance(r, dict) for r in result)
        assert all(expected_keys <= set(r.keys()) for r in result), \
            f'missing keys in row: {result[0].keys()}'
        passed += 1; print(f'\u2705 Check 4: rows are dicts with correct keys')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: all values are strings (DictReader does NOT convert types)
    try:
        assert all(isinstance(r['amount'], str) for r in result if r['amount']), \
            'amount values should be strings after extract (no type conversion yet)'
        first = result[0]
        assert first['product'] == 'Laptop'
        assert first['amount'] == '999.99'   # string, not float
        passed += 1; print('\u2705 Check 5: values are strings (as expected from DictReader)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def extract(csv_text: str) -> list:
    reader = csv.DictReader(io.StringIO(csv_text))
    return list(reader)
```

</details>